# 4. Analysis 
This notebook uses networks and network statistics output from `3_NetworkExtraction.ipynb` notebook to conducts the core analyses of this paper:
* **Analysis 1: Network structure + personality**
* **Analysis 2: Self-similarity vs. ideological similarity**
Results are saved for visualizion in `5_Figures.ipynb`

Input:\
`data/network_data.csv` : a file with extracted networks (node + edge lists) and network statistics.

Output:\
`data/stats.csv` : Reshaped input data, so every _response_ is an observation. Original data has every _subject_ as an observation.\
`data/corr.txt` : Matrix of pairwise correlations between network measures and personality traits.\
`data/p_vals.txt` : Matrix of p-values associated with pairwise correlations.\
`data/distances.json` : Pairwise distances (Portrait Divergence) between networks.

In [1]:
# helpful packages
import pandas as pd
import numpy as np
import itertools as it
import json

# network analysis
import networkx as nx
import netrd  # note: requires numpy==1.x
dist_obj = netrd.distance.PortraitDivergence()

# linear models
from pymer4.models import lmer # requires r package lme4
import polars as pl
import ast
from sklearn.preprocessing import StandardScaler
scaler = StandardScaler()

# visualization
import matplotlib.pyplot as plt
import seaborn as sns

# Load data & organize variables
Each text question `q` (e.g. 'V161069') has multiple fields associated with it:
* `q` = raw text
* `q_clean` = cleaned text (with spell check)
* `q_doc` = version that was SpaCy doc (would need to be reconverted after data load)
* `q_network` = node/edge list to reconstruct network object
* `q_stats` = dict of network statistics

In [2]:
# organize lists of variable categories

text_qs = ['V161069', # PRE: Text- What is it that R likes about Democratic Pres cand
           'V161072', # PRE: Text- What is it that R dislikes about Democratic Pres cand
           'V161075', # PRE: Text- What is it that R likes about Republican Pres cand
           'V161078', # PRE: Text- What is it that R dislikes about Republican Pres cand
           'V161098', # PRE: Text- What does R like about Democratic party
           'V161101', # PRE: Text- What does R dislike about the Democratic party
           'V161104', # PRE: Text- What does R like about Republican party
           'V161106'  # PRE: Text- What does R dislike about the Republican party
          ]

net_cols = [f'{q}_network' for q in text_qs] # node/edge lists for question
stat_col = [f'{q}_stats' for q in text_qs] # network stats for each question


politics = ['leftright', #0 = strong Dem, 6=strong Rep
            'politicalinterest', # 0 = not at all interested, 3= v interested
            'politicalparticipation', # 0 = no protest or petition, 0.5 = one activity, 1 = both activities
           ]

personality = ['TIPI_extraversion',
               'TIPI_agreeableness', 
               'TIPI_conscientiousness',
               'TIPI_emotionalstability', 
               'TIPI_openness'
              ]

demographics = ['age', 
                'female', 
                'education']

# Unique UserId + all demographic/personality columns
cols = ['subj_id'] + politics + personality + demographics


# stored network statistics, calculated for each text q
stats = ['giant component', # Connectivity
         'k avg',           # Complexity
         'clustering',
         'density',
         'entropy',         # Hierarchy
         'dissortativity',
         'k std']



In [3]:
# helper function to extract saved stats as dictionary
def to_dict(x):
    try:
        y = ast.literal_eval(x)
        if isinstance(y, dict):  # Ensure it's actually a dictionary
            return y
    except(ValueError, SyntaxError):
        return None  # Or handle the error as needed
    

df = pd.read_csv('data/network_data.csv', 
                 converters=dict((col, to_dict) for col in stat_col + net_cols))

# compute word count
for q in text_qs:
    df[f'{q}_wordcount'] = df[f'{q}_clean'].fillna('').apply(lambda x: len(x.split()))
wordcount_cols = [f'{q}_wordcount' for q in text_qs]

# rename subject_id
df = df.rename(columns={'V160001':'subj_id'})

print(f'Data includes {len(df)} observations on {len(df.columns)} variables.')
print(f'\nRecorded variables are:\n{", ".join(df.columns)}')

Data includes 4270 observations on 62 variables.

Recorded variables are:
subj_id, V160001_orig, age, female, education, leftright, politicalinterest, politicalparticipation, TIPI_extraversion, TIPI_agreeableness, TIPI_conscientiousness, TIPI_emotionalstability, TIPI_openness, V161069, V161072, V161075, V161078, V161098, V161101, V161104, V161106, party_id, V161069_clean, V161072_clean, V161075_clean, V161078_clean, V161098_clean, V161101_clean, V161104_clean, V161106_clean, V161069_doc, V161072_doc, V161075_doc, V161078_doc, V161098_doc, V161101_doc, V161104_doc, V161106_doc, V161069_network, V161072_network, V161075_network, V161078_network, V161098_network, V161101_network, V161104_network, V161106_network, V161069_stats, V161072_stats, V161075_stats, V161078_stats, V161098_stats, V161101_stats, V161104_stats, V161106_stats, V161069_wordcount, V161072_wordcount, V161075_wordcount, V161078_wordcount, V161098_wordcount, V161101_wordcount, V161104_wordcount, V161106_wordcount


# Analysis 1: Network structure + personality
1. Load data with network statistics for each response
2. Multilevel model: $s = \beta p + \alpha_q + wc + \epsilon$. Captures relationship between personal trait $p$ and network statistic $s$ while controlling for question-level random effects $\alpha_q$ and word count $wc$. Cluster standard errors as respondent level
3. Save t-statistics if signficant -- these are the correlations of interest

In [4]:
# reshape data to one line per response (from one line per respondent)

# extract network data
data_stats = df[cols + stat_col].melt(id_vars=cols, value_vars=stat_col, 
                                var_name='question',
                                value_name='net_stats')

# extract word counts
data_wc = df[cols + wordcount_cols].melt(id_vars=cols, value_vars=wordcount_cols,
                                   var_name='question', 
                                   value_name='word_count')

data_wc['question'] = data_wc['question'].str.replace('_wordcount', '_stats') # align question key

# merge stats and word counts together
data = data_stats.merge(data_wc[['subj_id', 'question', 'word_count']], 
                         on=['subj_id', 'question'])


# drop rows with no network stats
data = data[data['net_stats'].apply(lambda x: isinstance(x, dict) and len(x)>0)]

# unwrap stats
for stat in stats:    
    data[stat] = data['net_stats'].apply(lambda x: x[stat])
    
# save to file
data.to_csv('data/stats.csv', index=False)

## Fit Multi-level Models & Extract t-values

In [5]:
def get_tvals(measures, stats, data):
    t_matrix = np.zeros((len(measures), len(stats)))
    p_matrix = np.zeros((len(measures), len(stats)))
    
    all_results = []

    # for each (personal trait, network measure) pair
    for measure_index, net_index in list(it.product(range(len(measures)), range(len(stats)))):
        measure_stat = measures[measure_index] # personal trait: politics, personality, or demographics
        net_stat = stats[net_index] # network statistic

        # create a smaller dataframe
        df = data[['subj_id', 'question', measure_stat, net_stat, 'word_count']]
        df = df.rename({measure_stat: 'measure_stat', net_stat: 'net_stat'})
        df = df.drop_nans()

        # run model
        model = lmer('net_stat ~ measure_stat  + word_count + (1 | question)', data=df)
        model.fit(no_warnings=True, 
                  summarize=False, 
                  verbose = False,
                  cluster = 'subj_id') # cluster SE by subject_id           

        ### get t-val
        t_val = model.result_fit['t_stat'][1]

        if np.isnan(t_val):
            t_val = 0
            print(f'Warning: no t_val found for trait {measure_stat}, network stat {net_stat}.\
                Correlation estimated at 0.')
        
        # record t-value
        t_matrix[measure_index][net_index] = t_val

        ### get p-val
        p_val = model.result_fit['p_value'][1]
        p_matrix[measure_index][net_index] = p_val
        
        # save model summary
        results = model.result_fit.to_pandas()
        results['personal_trait']=measure_stat
        results['network_measure']=net_stat
        all_results.append(results)
        
    corr = pd.DataFrame(t_matrix.T, index=stats, columns=measures)
    p = pd.DataFrame(p_matrix.T, index=stats, columns=measures)    
    results_df = pd.concat(all_results)

    return corr, p, results_df

In [6]:
# columns we'll need later
keep = data[['subj_id', 'question']]

# personal measures
measures = politics + personality + demographics

# columns to rescale
rescale = measures + stats + ['word_count']

# subset date to columns to rescale
data = data[rescale]

# rescale data
scaled = pd.DataFrame(scaler.fit_transform(data), 
                      columns=rescale, index=data.index)

data = pd.concat([keep, scaled], axis=1)

In [7]:
corr, p, results_df = get_tvals(measures, stats, pl.DataFrame(data))

In [8]:
corr.to_csv('data/regressions/corr.txt')
p.to_csv('data/regressions/p_vals.txt')
results_df.to_csv('data/regressions/full_results.csv', index=False)

### Word count correlations

While the above models control for word count, we'll also check the correlations between word count and traits. This model helps us see what we get from word count vs. what we get from network structure. 

In this model, we test: $s = wc + \alpha_q + \epsilon$ -- Capturing the relationship between personal trait $p$ and word count $wc$ while controlling for question-level random effects $\alpha_q$. Cluster standard errors as respondent level

In [9]:
# for each (personal trait, network measure) pair
t_matrix = np.zeros(len(measures))
p_matrix = np.zeros(len(measures))

all_results = []

for measure_index in range(len(measures)):
    measure_stat = measures[measure_index] # personal trait: politics, personality, or demographics
 
    # create a smaller dataframe
    df = pl.DataFrame(data[['subj_id', 'question', measure_stat, 'word_count']])
    df = df.rename({measure_stat: 'measure_stat'})
    df = df.drop_nans()

    # run model
    model = lmer('word_count ~ measure_stat + (1 | question)', data=df)
    model.fit(no_warnings=True, 
              summarize=False, 
              verbose = False,
              cluster = 'subj_id') # cluster SE by subject_id           

    ### get t-val
    t_val = model.result_fit['t_stat'][1]

    if np.isnan(t_val):
        t_val = 0
        print(f'Warning: no t_val found for trait {measure_stat}')
              
    # record t-value
    t_matrix[measure_index] = t_val

    ### get p-val
    p_val = model.result_fit['p_value'][1]
    p_matrix[measure_index] = p_val

    # save all relevant measures
    results = model.result_fit.to_pandas()
    results['personal_trait']=measure_stat
    all_results.append(results)

        
corr_wc = pd.DataFrame(t_matrix, index=measures).T
p_wc = pd.DataFrame(p_matrix, index=measures).T
results_df = pd.concat(all_results)

In [10]:
corr_wc.to_csv('data/regressions/corr_wc.txt')
p_wc.to_csv('data/regressions/p_vals_wc.txt')
results_df.to_csv('data/regressions/full_results_wc.csv', index=False)

# Analysis 2: Self-similarity vs. ideological similarity

For each respondent:

* Identify qs which are ideologically aligned v. not (ie: if Dem, likes about Dem cand)
* Compare similarity of network structure between aligned v. not aligned qs
* Identify comparison set of users with same ideology
* Compare similarity of network structure between same qs

## Reshape data

In [22]:
# reshape data
cols = ['subj_id', 'leftright']

data = df[cols + net_cols].melt(id_vars=cols, value_vars=net_cols, 
                                var_name='question',
                                value_name='network')

# drop rows with no nodes / edges
data = data[data['network'].apply(lambda x: len(x['nodes']) > 1 and len(x['edges']) > 0)]
print(f'{len(data)} observations')

14222 observations


In [23]:
# reset index twice, rename to obs_id
data = data.reset_index()
data = data[data.columns[1:]] # drop old index col
data = data.reset_index()
data = data.rename(columns={'index': 'obs_id'}) # contiguous obs id

## Construct Networkx Objects for Each Network

In [24]:
def get_network(network):
    nodes = network['nodes']
    edges = network['edges']
    
    G = nx.Graph()
    G.add_weighted_edges_from(edges)

    for node in nodes:
        G.add_node(node)

    return G

In [25]:
n = len(df) # num unique networks
networks = dict() # {obs_id : G}

for i, row in data.iterrows():
    G = get_network(row['network'])

    # save to dict
    networks[i] = G

## Get Comparison Networks for Each Respondent

In [26]:
# for each SUBJECT determine self + comparison networks
comparisons = dict()

for subj in set(data['subj_id']):
    comparisons.setdefault(subj, dict())
    
    ##### Self data ####
    self_data = data[data['subj_id']==subj]
    self_obs = self_data['obs_id'].values # keys for my own networks
    
    # save self data
    comparisons[subj]['self'] = self_obs

    ##### Comparisons  ####
    
    # my party id
    pid = self_data['leftright'].values[0]
    
    # the questions I answered
    qs = self_data['question'] # {obs_id : q}

    # responses from same party_id, but not me
    party_sub = data[(data['leftright']==pid) & (data['subj_id']!=subj)]
    
    # for each question
    for i, q in qs.items():
        comparisons[subj].setdefault('comp', dict())
        
        # obs_id of comparison networks
        comp = party_sub[party_sub['question']==q]['obs_id'].values
        
        # {my_network_for_q : comparison_networks_for_same_q}
        comparisons[subj]['comp'][i] = comp

print(f'{len(comparisons)} unique subjects found')

3790 unique subjects found


## Calculate Distances

In [27]:
distances = dict()

for count, subj in enumerate(comparisons):
    if count%1000==0:
        print(f'Processing subject #{count}')
    
    distances.setdefault(subj, dict())
    
    #### self networks
    nets = comparisons[subj]['self']
    N = len(nets)
    combs = int((N * (N - 1)) / 2)
    dists = np.zeros(combs)
    
    for x, (i, j) in enumerate(it.combinations(nets, 2)):
        # network i
        G = networks[i]

        # network j
        H = networks[j]

        # calculate distance
        dist = dist_obj.dist(G, H)
        
        # save distance
        dists[x] = dist
    
    # save all self-distances
    distances[subj]['self'] = dists
    
    ###### Comparison networks ####
    net_dict = comparisons[subj]['comp']
    dist_lists = list()
    
    for i, comps in net_dict.items():
        N = len(comps)
        combs = int((N * (N - 1)) / 2)
        q_dists = np.zeros(combs)
        
        # network i (subj network)
        G = networks[i]
        
        # network j (network from user w same pid, same q)
        for x, j in enumerate(comps):
            H = networks[j]
            
            # calculate distance
            dist = dist_obj.dist(G, H)
            
            # save distances for this q
            q_dists[x] = dist
        
        # append dists to this question
        dist_lists.append(q_dists)
        
    # save all comparison distances
    dists = np.concatenate(dist_lists)
    distances[subj]['comp'] = dists

Processing subject #0
Processing subject #1000
Processing subject #2000
Processing subject #3000


In [28]:
conv = dict()

for k, vs in distances.items():
    conv.setdefault(k, dict())
    
    for mode, v in vs.items():
        conv[k][mode] = list(v)

In [29]:
with open('data/distances.json', 'w') as fp:
    fp.write(json.dumps(conv))